In [0]:
from pyspark.sql.functions import *
from pyspark.sql import Window

inventory = spark.table(
    "agentdb.gold.fact_inventory_snapshot"
)

In [0]:
forecast = spark.table(
    "agentdb.forecasting.forecast_predictions"
)
active_model = (
    spark.table(
        "agentdb.forecasting.forecast_model_registry"
    )
    .filter(
        col("is_active") == True
    )
    .select("model_name")
    .first()[0]
)
forecast = (
    forecast
    .filter(
        col("model_name") == active_model
    )
)

In [0]:
forecast_summary = (

    forecast

    .groupBy(
        "product_key",
        "store_key"
    )

    .agg(

        avg(
            "predicted_demand"
        ).alias(
            "forecast_daily_demand"
        ),

        sum(
            "predicted_demand"
        ).alias(
            "forecast_30d"
        )
    )

    .withColumn(
        "forecast_7d",
        col(
            "forecast_daily_demand"
        ) * 7
    )

    .withColumn(
        "forecast_14d",
        col(
            "forecast_daily_demand"
        ) * 14
    )
)

In [0]:
inventory_risk = (

    inventory
    .withColumn(
        "inventory_qty",
        col("inventory_qty").cast("double")
    )

    .join(
        forecast_summary,
        [
            "product_key",
            "store_key"
        ]
    )

    .withColumn(
        "projected_days_to_stockout",

        when(
            col(
                "forecast_daily_demand"
            ) > 0,

            col(
                "inventory_qty"
            )
            /
            col(
                "forecast_daily_demand"
            )
        )
    )

    .withColumn(
        "safety_stock",

        col(
            "forecast_daily_demand"
        ) * 7
    )

    .withColumn(

        "risk_level",

        when(
            col(
                "projected_days_to_stockout"
            ) < 7,
            "CRITICAL"
        )

        .when(
            col(
                "projected_days_to_stockout"
            ) < 14,
            "HIGH"
        )

        .when(
            col(
                "projected_days_to_stockout"
            ) < 30,
            "MEDIUM"
        )

        .otherwise(
            "LOW"
        )
    )

    .withColumn(
        "created_at",
        current_timestamp()
    )
)

In [0]:
(
    inventory_risk
    .select(
        "product_key",
        "store_key",
        "inventory_qty",
        "forecast_daily_demand",
        "forecast_7d",
        "forecast_14d",
        "forecast_30d",
        "days_of_cover",
        "projected_days_to_stockout",
        "safety_stock",
        "risk_level",
        "created_at"
    )
    .write
    .mode("overwrite")
    .saveAsTable(
        "agentdb.intelligence.inventory_risk"
    )
)

In [0]:
display(
    inventory_risk
    .groupBy(
        "risk_level"
    )
    .count()
)